# Fine-Tuning / PEFT Component - GenAI Mentor

This notebook completes the fine-tuning / PEFT component of the GENAI course project from start to finish.

We use **LoRA** instead of full fine-tuning because LoRA is parameter-efficient, requires much less memory, and updates only small adapter matrices instead of all model weights.

**Base model:** `Qwen/Qwen2.5-0.5B-Instruct`

This model is small, instruction-tuned, compatible with Hugging Face Transformers, and suitable for LoRA fine-tuning on limited compute such as Apple MPS.

**Goal:** adapt the model into a GenAI course assistant that can answer as a tutor, examiner, or critic using the project fine-tuning dataset.

## 1. Install / Import Libraries

If any package is missing, uncomment the `%pip install` line and rerun the cell.

In [ ]:
# %pip install -q transformers datasets peft accelerate torch pandas numpy tqdm

import inspect
import json
import math
import os
import random
import shutil
import sys
from pathlib import Path

import accelerate
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from peft import LoraConfig, PeftModel, get_peft_model
from torch.nn.utils.rnn import pad_sequence
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainerCallback, TrainingArguments

print('Imports loaded successfully.')
print('accelerate:', accelerate.__version__)
print('torch:', torch.__version__)

## 2. Configuration

This notebook uses the final Qwen adapter path. Cleanup is intentionally separated into a manual opt-in cell so rerunning the notebook never deletes completed training artifacts by accident.


In [ ]:
ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
OUTPUT_DIR = ROOT / 'outputs' / 'finetune'
ADAPTER_DIR = OUTPUT_DIR / 'qwen_0_5b_lora_adapter'
RESULTS_DIR = OUTPUT_DIR / 'results'
DATA_DIR = ROOT / 'data' / 'finetune'

MAX_SEQ_LENGTH = 768
LEARNING_RATE = 1e-5
EPOCHS = 2
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
MAX_GRAD_NORM = 0.3
MAX_TOTAL_EXAMPLES = 1000
SEED = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', ROOT)
print('Base model:', BASE_MODEL)
print('Adapter directory:', ADAPTER_DIR)
print('Results directory:', RESULTS_DIR)


### Optional Manual Cleanup

Set `RUN_CLEANUP = True` only when you intentionally want to delete the final adapter/checkpoints and retrain from scratch. It is `False` by default to protect completed training outputs.


In [ ]:
RUN_CLEANUP = False

if RUN_CLEANUP:
    for path in [ADAPTER_DIR, OUTPUT_DIR / 'qwen_0_5b_checkpoints']:
        if path.exists():
            print('Deleting:', path)
            shutil.rmtree(path)
    print('Cleanup complete. Rerun training cells to recreate artifacts.')
else:
    print('Cleanup skipped. Existing adapter/checkpoints are preserved.')


## 3. Device Detection

CUDA is preferred when available. On Apple Silicon MPS, mixed precision is disabled and pin memory is disabled for stability.

In [ ]:
cuda_available = torch.cuda.is_available()
mps_available = hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()

if cuda_available:
    DEVICE = 'cuda'
elif mps_available:
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

USE_FP16 = DEVICE == 'cuda'
USE_BF16 = False
PIN_MEMORY = False if DEVICE == 'mps' else True

print('CUDA available:', cuda_available)
print('MPS available:', mps_available)
print('Selected device:', DEVICE)
print('fp16:', USE_FP16)
print('bf16:', USE_BF16)
print('pin_memory:', PIN_MEMORY)

if DEVICE == 'cpu':
    print('Warning: CPU training will be very slow. Use CUDA or MPS for the final run.')

## 4. Dataset Loading And Preparation

This cell loads the project SFT examples, cleans malformed or weak examples, converts every row into `instruction`, `input`, `output`, and `messages`, removes duplicates, and writes an 80/10/10 split.

In [ ]:
SYSTEM_PROMPT = 'You are a helpful adaptive GenAI course tutor. Explain concepts clearly, use course terminology, and avoid unsupported claims.'

PREFERRED_DATASET_PATHS = [
    DATA_DIR / 'sft_chat_dataset.jsonl',
    DATA_DIR / 'combined_dataset_clean.jsonl',
]
COMMON_DATASET_DIRS = [
    DATA_DIR,
    ROOT / 'data' / 'finetuning',
    ROOT / 'data',
    ROOT / 'notebooks',
]


def read_jsonl(path: Path):
    rows = []
    if not path.exists():
        return rows
    with path.open('r', encoding='utf-8') as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                print(f'Skipping malformed JSONL line {line_number} in {path}')
    return rows


def write_jsonl(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as file:
        for row in rows:
            file.write(json.dumps(row, ensure_ascii=False) + '\n')


def row_has_training_fields(row: dict) -> bool:
    if not isinstance(row, dict):
        return False
    messages = row.get('messages')
    if isinstance(messages, list):
        roles = {message.get('role') for message in messages if isinstance(message, dict)}
        if {'user', 'assistant'} <= roles:
            return True
    has_instruction = bool(str(row.get('instruction', '')).strip())
    has_output = bool(str(row.get('output') or row.get('response') or row.get('answer') or '').strip())
    return has_instruction and has_output


def jsonl_has_training_schema(path: Path, sample_size: int = 80) -> bool:
    checked = 0
    if not path.exists() or path.suffix != '.jsonl':
        return False
    try:
        with path.open('r', encoding='utf-8') as file:
            for line in file:
                line = line.strip()
                if not line:
                    continue
                checked += 1
                try:
                    row = json.loads(line)
                except json.JSONDecodeError:
                    continue
                if row_has_training_fields(row):
                    return True
                if checked >= sample_size:
                    break
    except OSError:
        return False
    return False


def discover_dataset_path() -> Path:
    for path in PREFERRED_DATASET_PATHS:
        if jsonl_has_training_schema(path):
            print('Using preferred dataset:', path)
            return path

    candidates = []
    for directory in COMMON_DATASET_DIRS:
        if not directory.exists():
            continue
        for path in directory.rglob('*.jsonl'):
            if path in PREFERRED_DATASET_PATHS:
                continue
            if jsonl_has_training_schema(path):
                name = path.name.lower()
                priority = (
                    0 if 'sft' in name else
                    1 if 'combined' in name and 'clean' in name else
                    2 if 'clean' in name else
                    3 if name in {'train.jsonl', 'val.jsonl', 'test.jsonl'} else
                    4
                )
                candidates.append((priority, path.stat().st_size, path))

    if not candidates:
        searched = ', '.join(str(path) for path in COMMON_DATASET_DIRS)
        raise FileNotFoundError(
            'No JSONL dataset with `messages` or `instruction` + `output/response/answer` fields was found. '
            f'Searched: {searched}'
        )

    candidates.sort(key=lambda item: (item[0], -item[1], str(item[2])))
    selected = candidates[0][2]
    print('Using discovered dataset:', selected)
    return selected


def extract_instruction_input(user_text: str):
    instruction = ''
    input_text = user_text
    if 'Instruction:' in user_text:
        after_instruction = user_text.split('Instruction:', 1)[1]
        if 'Input:' in after_instruction:
            instruction, input_text = after_instruction.split('Input:', 1)
        else:
            instruction = after_instruction
    return instruction.strip(), input_text.strip()


def normalize_row(row):
    if isinstance(row.get('messages'), list) and len(row['messages']) >= 2:
        messages = row['messages']
        system = next((m.get('content', '') for m in messages if m.get('role') == 'system'), SYSTEM_PROMPT)
        user = next((m.get('content', '') for m in messages if m.get('role') == 'user'), '')
        assistant = next((m.get('content', '') for m in messages if m.get('role') == 'assistant'), '')
        instruction, input_text = extract_instruction_input(user)
    else:
        system = SYSTEM_PROMPT
        instruction = str(row.get('instruction', '')).strip()
        input_text = str(row.get('input') or row.get('context') or '').strip()
        assistant = str(row.get('output') or row.get('response') or row.get('answer') or '').strip()
        user = f'Instruction: {instruction}\nInput:\n{input_text}'

    if not instruction:
        instruction = 'Respond as an adaptive GenAI course assistant.'
    normalized = {
        'instruction': instruction,
        'input': input_text,
        'output': assistant,
        'messages': [
            {'role': 'system', 'content': system or SYSTEM_PROMPT},
            {'role': 'user', 'content': f'Instruction: {instruction}\nInput:\n{input_text}'},
            {'role': 'assistant', 'content': assistant},
        ],
    }
    return normalized


source_path = discover_dataset_path()
raw_rows = read_jsonl(source_path)
print('Raw examples:', len(raw_rows))

clean_rows = []
seen = set()
for row in raw_rows:
    item = normalize_row(row)
    output = item['output'].strip()
    user_content = item['messages'][1]['content'].strip()
    if not user_content or not output:
        continue
    if len(output.split()) < 12:
        continue
    key = (user_content.lower(), output.lower())
    if key in seen:
        continue
    seen.add(key)
    clean_rows.append(item)

random.seed(SEED)
random.shuffle(clean_rows)
if MAX_TOTAL_EXAMPLES and len(clean_rows) > MAX_TOTAL_EXAMPLES:
    clean_rows = clean_rows[:MAX_TOTAL_EXAMPLES]

n = len(clean_rows)
train_end = int(n * 0.8)
val_end = int(n * 0.9)
train_rows = clean_rows[:train_end]
val_rows = clean_rows[train_end:val_end]
test_rows = clean_rows[val_end:]

assert len(train_rows) >= 300, f'Training set too small: {len(train_rows)}'

write_jsonl(DATA_DIR / 'train.jsonl', train_rows)
write_jsonl(DATA_DIR / 'val.jsonl', val_rows)
write_jsonl(DATA_DIR / 'test.jsonl', test_rows)

dataset_stats = {
    'source_path': str(source_path),
    'raw_examples': len(raw_rows),
    'clean_examples_used': n,
    'train_examples': len(train_rows),
    'validation_examples': len(val_rows),
    'test_examples': len(test_rows),
    'split': '80/10/10',
    'base_model': BASE_MODEL,
}
(DATA_DIR / 'dataset_stats.json').write_text(json.dumps(dataset_stats, indent=2), encoding='utf-8')
dataset_stats


## 5. Show Dataset Examples

In [ ]:
for index, example in enumerate(train_rows[:3], start=1):
    print('=' * 100)
    print(f'Example {index}')
    print('Instruction:', example['instruction'][:500])
    print('\nInput:', example['input'][:700])
    print('\nOutput:', example['output'][:900])

## 6. Load Tokenizer And Base Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_dtype = torch.float16 if DEVICE == 'cuda' else torch.float32
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=model_dtype)
base_model.to(DEVICE)

print('Tokenizer pad token:', tokenizer.pad_token)
print('Model loaded on:', DEVICE)

## 7. Preprocessing / Tokenization

The labels mask the system/user prompt and train only on assistant-style response tokens.

In [ ]:
def chat_text(messages, add_generation_prompt=False):
    if hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=add_generation_prompt)
    rendered = ''
    for message in messages:
        rendered += f"{message['role'].upper()}: {message['content']}\n"
    if add_generation_prompt:
        rendered += 'ASSISTANT: '
    return rendered

def tokenize_example(example):
    messages = example['messages']
    full_text = chat_text(messages, add_generation_prompt=False)
    prompt_text = chat_text(messages[:-1], add_generation_prompt=True)
    full = tokenizer(full_text, truncation=True, max_length=MAX_SEQ_LENGTH)
    prompt = tokenizer(prompt_text, truncation=True, max_length=MAX_SEQ_LENGTH)
    labels = list(full['input_ids'])
    prompt_len = min(len(prompt['input_ids']), len(labels))
    labels[:prompt_len] = [-100] * prompt_len
    if all(label == -100 for label in labels):
        labels = list(full['input_ids'])
    full['labels'] = labels
    return full

dataset = DatasetDict({
    'train': Dataset.from_list(train_rows),
    'validation': Dataset.from_list(val_rows),
    'test': Dataset.from_list(test_rows),
})

remove_columns = dataset['train'].column_names
tokenized_dataset = dataset.map(tokenize_example, remove_columns=remove_columns)
tokenized_dataset

In [ ]:
class AssistantOnlyDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        input_ids = [torch.tensor(feature['input_ids'], dtype=torch.long) for feature in features]
        attention_mask = [torch.tensor(feature['attention_mask'], dtype=torch.long) for feature in features]
        labels = [torch.tensor(feature['labels'], dtype=torch.long) for feature in features]
        return {
            'input_ids': pad_sequence(input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id),
            'attention_mask': pad_sequence(attention_mask, batch_first=True, padding_value=0),
            'labels': pad_sequence(labels, batch_first=True, padding_value=-100),
        }

data_collator = AssistantOnlyDataCollator(tokenizer)

## 8. LoRA Configuration

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
total_params = sum(param.numel() for param in model.parameters())
print(f'Trainable params: {trainable_params:,}')
print(f'Total params: {total_params:,}')
print(f'Trainable percent: {100 * trainable_params / total_params:.4f}%')

## 9. Training Arguments

The callback stops training if a logged loss becomes NaN or infinite.

In [ ]:
class StopOnBadLossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        logs = logs or {}
        for key in ['loss', 'eval_loss']:
            if key in logs and not math.isfinite(float(logs[key])):
                print(f'Stopping training because {key} is not finite: {logs[key]}')
                control.should_training_stop = True

training_kwargs = {
    'output_dir': str(OUTPUT_DIR / 'qwen_0_5b_checkpoints'),
    'learning_rate': LEARNING_RATE,
    'num_train_epochs': EPOCHS,
    'per_device_train_batch_size': BATCH_SIZE,
    'per_device_eval_batch_size': BATCH_SIZE,
    'gradient_accumulation_steps': GRAD_ACCUM_STEPS,
    'max_grad_norm': MAX_GRAD_NORM,
    'warmup_ratio': 0.03,
    'weight_decay': 0.01,
    'logging_steps': 5,
    'eval_steps': 25,
    'save_steps': 25,
    'save_total_limit': 2,
    'fp16': USE_FP16,
    'bf16': USE_BF16,
    'dataloader_pin_memory': PIN_MEMORY,
    'report_to': [],
}

signature = inspect.signature(TrainingArguments.__init__)
if 'eval_strategy' in signature.parameters:
    training_kwargs['eval_strategy'] = 'steps'
else:
    training_kwargs['evaluation_strategy'] = 'steps'

training_args = TrainingArguments(**training_kwargs)
training_args

## 10. Train LoRA Adapter

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    data_collator=data_collator,
    callbacks=[StopOnBadLossCallback()],
)

train_result = trainer.train()
train_metrics = train_result.metrics
eval_metrics = trainer.evaluate()

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

(OUTPUT_DIR / 'train_metrics.json').write_text(json.dumps(train_metrics, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'eval_metrics.json').write_text(json.dumps(eval_metrics, indent=2), encoding='utf-8')

training_log = {
    'status': 'completed',
    'base_model': BASE_MODEL,
    'adapter_dir': str(ADAPTER_DIR),
    'device': DEVICE,
    'dataset_stats': dataset_stats,
    'lora': {'r': 8, 'lora_alpha': 16, 'lora_dropout': 0.05},
    'train_metrics': train_metrics,
    'eval_metrics': eval_metrics,
    'log_history': trainer.state.log_history,
}
(OUTPUT_DIR / 'training_log.json').write_text(json.dumps(training_log, indent=2), encoding='utf-8')

print('Adapter saved to:', ADAPTER_DIR)
print('Train metrics:', train_metrics)
print('Eval metrics:', eval_metrics)

## 11. Display Training Logs

In [ ]:
log_df = pd.DataFrame(trainer.state.log_history)
display_columns = [column for column in ['epoch', 'step', 'loss', 'eval_loss', 'learning_rate'] if column in log_df.columns]
log_df[display_columns].dropna(how='all').tail(30)

## 12. Base Model Inference Function

The base evaluation model is loaded once and reused for every evaluation/demo prompt. It is released only after evaluation and demo comparison are complete.


In [ ]:
def clear_device_cache():
    if DEVICE == 'mps':
        torch.mps.empty_cache()
    elif DEVICE == 'cuda':
        torch.cuda.empty_cache()


# Training logs were already displayed in the previous cell. Release training objects before evaluation.
for object_name in ['trainer', 'model', 'base_model']:
    if object_name in globals():
        del globals()[object_name]
clear_device_cache()

base_eval_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=model_dtype)
base_eval_model.to(DEVICE)
base_eval_model.eval()
print('Base evaluation model loaded once for all evaluation/demo prompts.')


def render_prompt(prompt: str):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt},
    ]
    return chat_text(messages, add_generation_prompt=True)


def generate_base_answer(prompt: str, max_new_tokens: int = 180) -> str:
    inputs = tokenizer([render_prompt(prompt)], return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        outputs = base_eval_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated = outputs[0][inputs.input_ids.shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


## 13. Fine-Tuned Model Inference Function

In [ ]:
tuned_base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=model_dtype)
tuned_base_model.to(DEVICE)
tuned_model = PeftModel.from_pretrained(tuned_base_model, ADAPTER_DIR)
tuned_model.eval()

def generate_tuned_answer(prompt: str, max_new_tokens: int = 180) -> str:
    inputs = tokenizer([render_prompt(prompt)], return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        outputs = tuned_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated = outputs[0][inputs.input_ids.shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

print('Fine-tuned adapter loaded from:', ADAPTER_DIR)

## 14. Evaluate Base vs Fine-Tuned Model

In [ ]:
GENAI_KEYWORDS = {
    'rag', 'retrieval', 'embedding', 'transformer', 'attention', 'token', 'prompt', 'lora', 'fine-tuning',
    'adapter', 'hallucination', 'evaluation', 'grounded', 'context', 'course', 'model', 'llm'
}
UNSUPPORTED_FLAGS = {'medical diagnosis', 'guaranteed cure', 'hack', 'exam answers', 'ignore rules'}

def token_set(text: str):
    return {token.strip('.,:;!?()[]{}').lower() for token in text.split() if token.strip()}

def token_overlap_score(expected: str, actual: str):
    expected_tokens = token_set(expected)
    actual_tokens = token_set(actual)
    if not expected_tokens:
        return 0.0
    return len(expected_tokens & actual_tokens) / len(expected_tokens)

def keyword_coverage_score(actual: str):
    actual_lower = actual.lower()
    found = [keyword for keyword in GENAI_KEYWORDS if keyword in actual_lower]
    return min(1.0, len(found) / 5), found

def hallucination_flag(actual: str):
    actual_lower = actual.lower()
    return any(flag in actual_lower for flag in UNSUPPORTED_FLAGS)

def score_answer(expected: str, actual: str):
    overlap = token_overlap_score(expected, actual)
    keyword_score, keywords = keyword_coverage_score(actual)
    length_ok = 30 <= len(actual.split()) <= 260
    flagged = hallucination_flag(actual)
    score = 0.45 * overlap + 0.35 * keyword_score + 0.20 * float(length_ok) - 0.25 * float(flagged)
    return max(0.0, round(score, 3)), keywords, flagged, length_ok

EVAL_LIMIT = 25
comparison_rows = []
for example in tqdm(test_rows[:EVAL_LIMIT]):
    prompt = f"Instruction: {example['instruction']}\nInput:\n{example['input']}"
    expected = example['output']
    base_output = generate_base_answer(prompt, max_new_tokens=160)
    tuned_output = generate_tuned_answer(prompt, max_new_tokens=160)
    base_score, base_keywords, base_flag, base_length_ok = score_answer(expected, base_output)
    tuned_score, tuned_keywords, tuned_flag, tuned_length_ok = score_answer(expected, tuned_output)
    comparison_rows.append({
        'instruction': example['instruction'],
        'input': example['input'],
        'expected_output': expected,
        'base_model_output': base_output,
        'tuned_model_output': tuned_output,
        'base_score': base_score,
        'tuned_score': tuned_score,
        'base_keywords': ', '.join(base_keywords),
        'tuned_keywords': ', '.join(tuned_keywords),
        'base_hallucination_flag': base_flag,
        'tuned_hallucination_flag': tuned_flag,
        'base_length_ok': base_length_ok,
        'tuned_length_ok': tuned_length_ok,
        'improvement_notes': 'Tuned score higher' if tuned_score > base_score else 'Base score equal or higher',
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_csv = RESULTS_DIR / 'base_vs_tuned_comparison.csv'
comparison_df.to_csv(comparison_csv, index=False)

evaluation_summary = {
    'num_examples': len(comparison_df),
    'average_base_score': round(float(comparison_df['base_score'].mean()), 3),
    'average_tuned_score': round(float(comparison_df['tuned_score'].mean()), 3),
    'average_improvement': round(float((comparison_df['tuned_score'] - comparison_df['base_score']).mean()), 3),
    'tuned_wins': int((comparison_df['tuned_score'] > comparison_df['base_score']).sum()),
    'base_wins_or_ties': int((comparison_df['tuned_score'] <= comparison_df['base_score']).sum()),
    'comparison_csv': str(comparison_csv),
}
(RESULTS_DIR / 'evaluation_summary.json').write_text(json.dumps(evaluation_summary, indent=2), encoding='utf-8')

evaluation_summary

## 15. Demo Comparison On Fixed Prompts

In [ ]:
demo_prompts = [
    'Explain RAG simply.',
    'Explain LoRA vs full fine-tuning.',
    'Why do LLMs hallucinate?',
    'Create a study plan for learning transformers.',
    'How does fine-tuning help a GenAI course assistant?',
]

for idx, prompt in enumerate(demo_prompts, start=1):
    base_answer = generate_base_answer(prompt, max_new_tokens=180)
    tuned_answer = generate_tuned_answer(prompt, max_new_tokens=180)
    base_score, _, _, _ = score_answer(prompt, base_answer)
    tuned_score, _, _, _ = score_answer(prompt, tuned_answer)
    comparison = 'Fine-tuned answer scored higher by the simple project metric.' if tuned_score > base_score else 'Base answer was equal or higher on the simple project metric.'
    print('=' * 120)
    print(f'PROMPT {idx}: {prompt}')
    print('\nBASE MODEL ANSWER')
    print(base_answer)
    print('\nFINE-TUNED MODEL ANSWER')
    print(tuned_answer)
    print('\nSHORT COMPARISON')
    print(comparison)

# Release evaluation models after all base-vs-tuned evaluation and demo prompts complete.
for object_name in ['base_eval_model', 'tuned_model', 'tuned_base_model']:
    if object_name in globals():
        del globals()[object_name]
clear_device_cache()
print('Evaluation models released after comparison/demo evaluation.')


## 16. Results Summary

After running the cells above, inspect `evaluation_summary`. The expected improvement is behavioral: the LoRA-adapted model should more consistently follow the GenAI assistant role and course response structure. If the tuned score is weak or inconsistent, the next improvement is to train on more examples, use more epochs, and add a larger held-out evaluation set.

Known limitations:
- The automatic metrics are simple and should be supplemented with human review.
- A small local MPS run proves the PEFT pipeline and adapter creation, but stronger quality requires more training examples and more evaluation prompts.
- The adapter is only as good as the synthetic/course-aligned dataset quality.

## 17. Report-Ready Fine-Tuning / PEFT Explanation

We used LoRA parameter-efficient fine-tuning instead of full fine-tuning because it requires less memory and updates only adapter weights. The base model was `Qwen/Qwen2.5-0.5B-Instruct`. The dataset consisted of GenAI course assistant instruction-response examples. The dataset was cleaned by removing malformed, duplicate, empty, and very short examples, then split into train, validation, and test sets using an 80/10/10 split. The model was trained on the train split, monitored using the validation split, and evaluated on the held-out test split. We compared the base model against the LoRA-adapted model to show behavioral changes after fine-tuning.

The LoRA configuration used `r=8`, `lora_alpha=16`, `lora_dropout=0.05`, `bias='none'`, and `task_type='CAUSAL_LM'`, targeting the Qwen attention and MLP projection modules: `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, and `down_proj`. Training used a conservative learning rate of `1e-5`, batch size `1`, gradient accumulation `8`, max gradient norm `0.3`, warmup ratio `0.03`, and weight decay `0.01`.

After training, the adapter was saved to `outputs/finetune/qwen_0_5b_lora_adapter`, training metrics were saved to `outputs/finetune/train_metrics.json`, evaluation metrics were saved to `outputs/finetune/eval_metrics.json`, and base-vs-tuned comparison results were saved to `outputs/finetune/results/base_vs_tuned_comparison.csv` and `outputs/finetune/results/evaluation_summary.json`.

When reporting final numbers, use the actual values from `data/finetune/dataset_stats.json`, `outputs/finetune/train_metrics.json`, and `outputs/finetune/eval_metrics.json` after executing this notebook.